# Post Processing

Before post processing, we have to create another index file. This helps in analysing different domains of the proteins, membranes, etc.

To make it easier to work with and to speed up the analysis, it is better to remove solvent (when we are not dealing with analysis related to water/ions). Solvent takes up a lot of space and can reduce the speed of post-processing.



# Creating index file

In this index_analysis.ndx file, we have different groups of atoms such as protein, membranes, transmembrane domain of the protein, etc. This helps to analyse the behaviour of different groups with each other.

In [ ]:
from MDAnalysis import Universe
from MDAnalysis.selections.gromacs import SelectionWriter

u = Universe("../prod.tpr")

with SelectionWriter('index_analysis.ndx', mode='w') as ndx:
    ndx.write(u.select_atoms('protein'), name='PROT')
    ndx.write(u.select_atoms('bynum 242:312'), name='Transmembrane')
    ndx.write(u.select_atoms('resname POPC'), name='MEM')
    ndx.write(u.select_atoms('not (resname W ION)'), name='not_SOL')

# Correction of PBC

During MD simulation, periodic boundary conditions are used such that molecules leaving one side of the box re-enter from the opposite side. This creates an effectively infinite system, which is desirable.

However, for certain analysis or visualisation, this can result in “broken” molecules or fragmented trajectories.

To fix this, we apply a correction to make the molecules whole again. We use a GROMACS tool called `trjconv` for this.

In [ ]:
%%bash 
module load GCC/14.2.0 GROMACS/2025.4-CUDA-12.8.0-SMP
gmx trjconv -f ../prod.xtc -s ../prod.tpr -o frame0.pdb -n index_analysis.ndx -center -pbc mol -ur compact -e 0 -conect <<< "Transmembrane not_SOL"
gmx trjconv -f ../prod.xtc -s ../prod.tpr -o prod_noW.xtc -n index_analysis.ndx -center -pbc mol -ur compact <<< "Transmembrane not_SOL"
gmx trjconv -f ../prod.xtc -s ../prod.tpr -o prod_noW.pdb -n index_analysis.ndx -center -pbc mol -ur compact <<< "Transmembrane not_SOL"

| File                    | Meaning                                                |
| ----------------------- | ------------------------------------------------------ |
| `-f ../prod.xtc`        | Raw production trajectory                              |
| `-s ../prod.tpr`        | Run input file (provides structure/topology reference) |
| `-o prod_noW.xtc`       | Output cleaned trajectory                              |
| `-n index_analysis.ndx` | Index file defining custom groups                      |
| `-center`              | Centers on a chosen group (here **Transmembrane**)     |
| `-pbc mol`             | Keeps whole molecules intact                           |
| `-ur compact`          | Minimizes box volume representation                    |

`<<< "Transmembrane not_SOL"`: This is bash input redirection, not a GROMACS option. It automatically provides answers to the interactive prompts from `trjconv` when it asks for group selections — here, **"Transmembrane"** for centering and **"not_SOL"** (all atoms except solvent) for output.

# Basic Analysis in "MD Analysis"


In [16]:
#LOAD NECESSARY MODULES
import os
import numpy as np
import pandas as pd
import MDAnalysis as mda
from MDAnalysis.analysis import rms
import matplotlib.pyplot as plt

We define the variables for the topology and trajectory files. Based on the files present in the directory, we will use the Martini .tpr and .xtc files.

In [17]:
# Selected topology and trajectory
topology_file = "frame0.pdb"
trajectory_file = "prod_noW.xtc"


We use MDAnalysis to load the simulation universe and print some basic information about the system.

In [ ]:
u = mda.Universe(topology_file, trajectory_file)
print(u)
print("Number of atoms/beads:", len(u.atoms))
print("Number of residues:", len(u.residues))
print("Number of frames:", len(u.trajectory))

## Define Selections

Based on the analysis above, we set the final residue ranges. We will use simple residue-based selections since MDAnalysis may not automatically recognize Martini beads as "protein".

In [19]:
ig_resid_range = "3:89"     # Edit here if needed
tm_resid_range = "108:128"  # Edit here if needed

ig_domain = u.select_atoms(f"resid {ig_resid_range}")
tm_helix = u.select_atoms(f"resid {tm_resid_range}")

print("Ig domain beads:", len(ig_domain), "residues:", len(ig_domain.residues))
print("TM helix beads:", len(tm_helix), "residues:", len(tm_helix.residues))
print("Ig residues:", ig_domain.residues.resids.min(), "to", ig_domain.residues.resids.max())
print("TM residues:", tm_helix.residues.resids.min(), "to", tm_helix.residues.resids.max())

Ig domain beads: 198 residues: 87
TM helix beads: 41 residues: 21
Ig residues: 3 to 89
TM residues: 108 to 128


## RMSD of Whole Protein

- RMSD is calculated for the whole protein.
- The first frame of the trajectory is used as the reference structure.
- The trajectory is aligned using the whole protein, meaning this reports internal structural stability rather than global tumbling or drift.

In [ ]:
rmsd_analysis = rms.RMSD(
    u,
    u,
    select="resid 1-138",
    ref_frame=0
)
rmsd_analysis.run()

rmsd_df = pd.DataFrame(
    rmsd_analysis.results.rmsd,
    columns=["frame", "time_ps", "rmsd_A"]
)
rmsd_df["time_ns"] = rmsd_df["time_ps"] / 1000.0

display(rmsd_df.head())

plt.figure(figsize=(7, 4))
plt.plot(rmsd_df["time_ns"], rmsd_df["rmsd_A"])
plt.xlabel("Time (ns)")
plt.ylabel("Protein RMSD (Å)")
plt.title("CADM1 Protein RMSD")
plt.tight_layout()
plt.show()

## Define Geometry Functions for Orientation Analysis

- The Z-axis is assumed to be the simulation box z-axis `[0, 0, 1]`.
- The Ig-domain axis is approximated as the vector between the centers of geometry of the lower and upper halves of the domain.
- The TM axis is approximated similarly, using the lower and upper halves of the TM helix.
- This provides a simple, robust, and interpretable metric for domain tilt without requiring full principal axis calculations.

In [21]:
def unit_vector(v):
    return v / np.linalg.norm(v)

def angle_between_vectors(v1, v2):
    v1 = unit_vector(v1)
    v2 = unit_vector(v2)
    cosang = np.dot(v1, v2)
    # We use abs(cosang) because an axis has arbitrary direction sign
    cosang = np.clip(abs(cosang), -1.0, 1.0)
    angle = np.degrees(np.arccos(cosang))
    return angle

def split_selection_by_resid(atomgroup):
    residues = atomgroup.residues
    resids = residues.resids
    midpoint = np.median(resids)
    lower = atomgroup.select_atoms(f"resid {resids.min()}:{int(midpoint)}")
    upper = atomgroup.select_atoms(f"resid {int(midpoint)+1}:{resids.max()}")
    return lower, upper

def axis_from_split(atomgroup):
    lower, upper = split_selection_by_resid(atomgroup)
    lower_center = lower.center_of_geometry()
    upper_center = upper.center_of_geometry()
    # Vector from lower half to upper half
    return upper_center - lower_center

## Calculate Angles Over Trajectory

Loop through the trajectory frames, extract the vectors, and calculate the relevant angles in degrees (bounded between 0 and 90).

In [22]:
records = []
z_axis = np.array([0.0, 0.0, 1.0])

for ts in u.trajectory:
    ig_axis = axis_from_split(ig_domain)
    tm_axis = axis_from_split(tm_helix)
    
    ig_vs_z = angle_between_vectors(ig_axis, z_axis) # membrane_normal)
    ig_vs_tm = angle_between_vectors(ig_axis, tm_axis)
    tm_vs_z = angle_between_vectors(tm_axis, z_axis) # membrane_normal)
    
    records.append({
        "frame": ts.frame,
        "time_ps": ts.time,
        "time_ns": ts.time / 1000.0,
        "ig_vs_z_deg": ig_vs_z,
        "ig_vs_tm_deg": ig_vs_tm,
        "tm_vs_z_deg": tm_vs_z,
    })

angles_df = pd.DataFrame(records)
#display(angles_df.head())

### Plot Angles

Visualise the time series of the angles. The limits are set from 0 to 90 degrees for straightforward interpretation.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(angles_df["time_ns"], angles_df["ig_vs_z_deg"], color="tab:blue")
plt.xlabel("Time (ns)")
plt.ylabel("Angle (degrees)")
plt.title("Ig-domain axis vs Z axis")
plt.ylim(0, 90)
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(angles_df["time_ns"], angles_df["ig_vs_tm_deg"], color="tab:orange")
plt.xlabel("Time (ns)")
plt.ylabel("Angle (degrees)")
plt.title("Ig-domain axis vs transmembrane helix axis")
plt.ylim(0, 90)
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(angles_df["time_ns"], angles_df["tm_vs_z_deg"], color="tab:green")
plt.xlabel("Time (ns)")
plt.ylabel("Angle (degrees)")
plt.title("Transmembrane helix axis vs membrane normal")
plt.ylim(0, 90)
plt.tight_layout()
plt.show()

In [ ]:
import py3Dmol

with open("prod_noW.pdb", "r") as f:
    traj = f.read()

view = py3Dmol.view(width=900, height=700)

# Load all models as trajectory frames
view.addModelsAsFrames(traj, "pdb")

# Martini bead representation
view.setStyle({
    "sphere": {
        "radius": 1.0
    }
})

view.zoomTo()



In [ ]:
# Play the trajectory
view.animate({
    "loop": "forward",
    "interval": 100  # milliseconds between frames
})

view.show()